# Retrieval Evaluation for Three Course Datasets

This notebook runs retrieval evaluation for the three indexed course datasets by directly calling `scripts/retrieval/test_retrieval.py`.

Default datasets:

- `eods`: Elements of Data Science
- `adl`: Applied Deep Learning
- `5703`: Statistical Inference

The notebook defaults to the 40-question evaluation files. For courses with a 20-question file, change `EVAL_SIZE` to `20`.

## 1. Setup

In [1]:
from __future__ import annotations

import os
import re
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "scripts" / "retrieval" / "test_retrieval.py").exists():
    ROOT = Path.cwd().parent

TEST_SCRIPT = ROOT / "scripts" / "retrieval" / "test_retrieval.py"
DATA_TEST_DIR = ROOT / "data" / "test"

print(f"Repository root: {ROOT}")
print(f"Evaluation script: {TEST_SCRIPT}")

Repository root: /Users/serenacyn03/AI_Course_Assistant
Evaluation script: /Users/serenacyn03/AI_Course_Assistant/scripts/retrieval/test_retrieval.py


## 2. Configuration

Change these values if you want to compare a different retrieval setup.

In [2]:
EVAL_SIZE = 40

COURSES = [
    {"course_id": "eods", "name": "Elements of Data Science"},
    {"course_id": "adl", "name": "Applied Deep Learning"},
    {"course_id": "5703", "name": "Statistical Inference"},
]

TARGET = "both"          # one of: atomic, semantic, both
METHOD = "dense_rerank"  # one of: bm25, dense, hybrid, dense_rerank
CANDIDATE_K = 4          # must be >= 4 because the evaluator reports k=2,3,4
VERBOSE = False          # set True to print top hits for every query

EXTRA_ARGS = []
# Example:
# EXTRA_ARGS = ["--faiss-weight", "1.0", "--bm25-weight", "1.0"]

## 3. Helper Functions

In [3]:
def eval_json_path(course_id: str, eval_size: int) -> Path:
    return DATA_TEST_DIR / f"{course_id}_retrieval_eval_{eval_size}.json"


def build_command(course_id: str, eval_json: Path) -> list[str]:
    cmd = [
        sys.executable,
        str(TEST_SCRIPT),
        "--course-id",
        course_id,
        "--eval-json",
        str(eval_json),
        "--target",
        TARGET,
        "--method",
        METHOD,
        "--candidate-k",
        str(CANDIDATE_K),
    ]
    if VERBOSE:
        cmd.append("--verbose")
    return cmd + EXTRA_ARGS


def parse_metrics(stdout: str, course_id: str, course_name: str) -> list[dict[str, object]]:
    rows = []
    pattern = re.compile(r"^(\d+)\t([0-9.]+)\t([0-9.]+)\t([0-9.]+)\t([0-9.]+)$")
    for line in stdout.splitlines():
        match = pattern.match(line.strip())
        if not match:
            continue
        k, recall, precision, mrr, ndcg = match.groups()
        rows.append(
            {
                "course_id": course_id,
                "course_name": course_name,
                "k": int(k),
                "recall": float(recall),
                "precision": float(precision),
                "mrr": float(mrr),
                "ndcg": float(ndcg),
            }
        )
    return rows


def run_evaluation(course: dict[str, str]) -> tuple[list[dict[str, object]], str]:
    course_id = course["course_id"]
    course_name = course["name"]
    eval_json = eval_json_path(course_id, EVAL_SIZE)
    if not eval_json.exists():
        raise FileNotFoundError(f"Evaluation file not found: {eval_json}")

    cmd = build_command(course_id, eval_json)
    print("Running:", " ".join(cmd))
    result = subprocess.run(
        cmd,
        cwd=ROOT,
        text=True,
        capture_output=True,
        check=False,
    )

    output = result.stdout.strip()
    if result.stderr.strip():
        output = output + "\n\n[stderr]\n" + result.stderr.strip()
    if result.returncode != 0:
        raise RuntimeError(f"Evaluation failed for {course_id}:\n{output}")

    return parse_metrics(result.stdout, course_id, course_name), output

## 4. Run Evaluation

In [4]:
if METHOD in {"dense", "hybrid", "dense_rerank"} and not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is required for dense, hybrid, or dense_rerank evaluation.")

all_rows = []
raw_outputs = {}

for course in COURSES:
    rows, output = run_evaluation(course)
    all_rows.extend(rows)
    raw_outputs[course["course_id"]] = output
    print(output)
    print("-" * 80)

metrics_df = pd.DataFrame(all_rows)
display(metrics_df)

Running: /Users/serenacyn03/miniconda3/envs/ai-course-assistant/bin/python /Users/serenacyn03/AI_Course_Assistant/scripts/retrieval/test_retrieval.py --course-id eods --eval-json /Users/serenacyn03/AI_Course_Assistant/data/test/eods_retrieval_eval_40.json --target both --method dense_rerank --candidate-k 4
Evaluated 40 queries | target=both | method=dense_rerank | eval_json=/Users/serenacyn03/AI_Course_Assistant/data/test/eods_retrieval_eval_40.json

k	recall	precision	mrr	ndcg
2	0.4229	0.6125	0.8500	0.6606
3	0.4688	0.4583	0.8667	0.5633
4	0.5104	0.3750	0.8667	0.5742
--------------------------------------------------------------------------------
Running: /Users/serenacyn03/miniconda3/envs/ai-course-assistant/bin/python /Users/serenacyn03/AI_Course_Assistant/scripts/retrieval/test_retrieval.py --course-id adl --eval-json /Users/serenacyn03/AI_Course_Assistant/data/test/adl_retrieval_eval_40.json --target both --method dense_rerank --candidate-k 4
Evaluated 40 queries | target=both | met

,course_id,course_name,k,recall,precision,mrr,ndcg
0,eods,Elements of Data Science,2,0.4229,0.6125,0.8500,0.6606
1,eods,Elements of Data Science,3,0.4688,0.4583,0.8667,0.5633
2,eods,Elements of Data Science,4,0.5104,0.3750,0.8667,0.5742
3,adl,Applied Deep Learning,2,0.4771,0.4875,0.6875,0.5699
4,adl,Applied Deep Learning,3,0.5604,0.3833,0.7042,0.5505
5,adl,Applied Deep Learning,4,0.6104,0.3187,0.7104,0.5769
6,5703,Statistical Inference,2,0.4167,0.6250,0.7500,0.6307
7,5703,Statistical Inference,3,0.4667,0.4667,0.7667,0.5179
8,5703,Statistical Inference,4,0.5167,0.3875,0.7729,0.5482


## 5. Side-by-Side Summary

In [5]:
summary = metrics_df.pivot_table(
    index=["course_id", "course_name"],
    columns="k",
    values=["recall", "precision", "mrr", "ndcg"],
)
display(summary.round(4))

mrr                    ndcg          \
k                                        2       3       4       2       3   
course_id course_name                                                        
5703      Statistical Inference     0.7500  0.7667  0.7729  0.6307  0.5179   
adl       Applied Deep Learning     0.6875  0.7042  0.7104  0.5699  0.5505   
eods      Elements of Data Science  0.8500  0.8667  0.8667  0.6606  0.5633   

                                           precision                  recall  \
k                                        4         2       3       4       2   
course_id course_name                                                          
5703      Statistical Inference     0.5482    0.6250  0.4667  0.3875  0.4167   
adl       Applied Deep Learning     0.5769    0.4875  0.3833  0.3187  0.4771   
eods      Elements of Data Science  0.5742    0.6125  0.4583  0.3750  0.4229   

                                                    
k                                        3       4  
course_id course_name                               
5703      Statistical Inference     0.4667  0.5167  
adl       Applied Deep Learning     0.5604  0.6104  
eods      Elements of Data Science  0.4688  0.5104